# Experiment: Better Augment

In this experiment, there is added white noise to the training samples in the augmentation pipeline. Some potential better augmentations are added as well.

In [ ]:
%reload_ext autoreload
%autoreload 2
%load_ext line_profiler

import os
from dataclasses import dataclass
from pathlib import Path
from pprint import pprint

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from albumentations.pytorch import ToTensorV2
from clearml import Task
from dotenv import load_dotenv
from torch import nn
from torch.utils.data import DataLoader

from mglyph_ml.dataset.glyph_dataset import GlyphDataset
from mglyph_ml.dataset.utils import load_split_samples, show_datasets
from mglyph_ml.experiment.run_config import RunConfigBase
from mglyph_ml.experiment.visualization import show_loss_vs_x_graph, show_truth_vs_pred_graph
from mglyph_ml.nn.glyph_regressor_binned import BinnedGlyphRegressor

# Hyperparameters

Every experiment can run using some set of parameters. These parameters can vary literally anything in the experiment. They can change the shape of the neural network, they can vary the training set, set the learning rate, choose an optimizer, or simply set a random number generator seed. The sky is the limit here.

Notice that this cell has the `parameters` tag associated with it. That tag is very important, because it acts as a 'hook' for a tool called Papermill. This tool injects custom parameters into the notebook and runs it. Injecting parameters is useful because it allows for the execution of series of experiments. We can run the notebook 20 times with different parameters, and analyze the results.

When the notebook is run, it exports important information to the [ClearML web server](https://app.clear.ml/), where the data is aggregated, and can be analyzed. Some parameters like `task_tag` and `task_name` refer to the ClearML Task properties.

*`scheduler_gamma`:*

The gamma is calculated using this formula: $\text{total\_multiplier}^{\frac{1}{\text{total\_steps}}}$. The `total_decrease` corresponds to the total multiplier of the initial learning rate that we wish to be applied, and `total_steps` represents the number of steps necessary to achieve this multiplier. So, for example, if we want our learning rate to decay by a factor of 10 in 1000 steps, we can use $\text{gamma} = \frac{1}{10}^\frac{1}{1000} = \sqrt[1000]{\frac{1}{10}}$.

In [ ]:
@dataclass(frozen=True)
class RunConfig(RunConfigBase):
    # The ClearML task tag (so we can easily filter this experiment in the ClearML web GUI).
    task_tag: str = "tag-2"

    # Where the dataset lies.
    # dataset_path: Path = Path("data/simple-star-20k-dual.mglyph")
    dataset_path: Path = Path("data/varying-star-dual-transparent-1k.mglyph")

    # This seed is used by every single RNG in the experiment to make it 100% reproducible.
    seed: int = 329

    # Size of a single batch that is processed in one step.
    batch_size: int = 64

    # We don't have epochs, we only use "steps". 1 step corresponds to the training of the NN on a single batch.
    max_steps: int = 10000

    # The start learning rate. It changes during the experiment because of the LR scheduler.
    learning_rate: float = 0.0005

    # Whether to report the task to ClearML or not. Set to True to stop reporting to ClearML.
    offline: bool = True

    # The number of CPU threads spawned to load the data from a PyTorch DataLoader.
    data_loader_num_workers: int = 8

    # In binned regression, this corresponds to the number of divisions of the [0.0; 100.0] interval.
    num_divisions: int = 5

    # How fast the learning rate decays.
    scheduler_gamma: float = 0.5
    # How many steps it takes to multiply the LR by gamma.
    scheduler_steps: int = 2000

    # enable or disable training data augmentation
    augment: bool = True

    # NN width scaling number (the larger the number the larger the NN)
    width_multiplier: float = 1 / 32

    # Optional explicit name; if None, it is derived from other fields below.
    task_name: str = f"Better Augment [seed={seed},lr={learning_rate}]"


# Here, we clear the global variables that have the same names as the fields in this RunConfig class.
RunConfig.clear_globals()

In [ ]:
# This is the global RunConfig instance. You can use c.xxx to access the `xxx` parameter.
c = RunConfig.from_globals()

Task.set_offline(c.offline)
task: Task = Task.init(project_name="mglyph-ml", task_name=c.task_name)
task.add_tags(c.task_tag)
task.connect(c)

load_dotenv()
pprint(c)

# Data Loading

The data is loaded from the dataset file from so-called _splits_. You can think of a split like a folder inside the dataset; it's simply used to group a bunch of samples together. A single sample inside the dataset has these properties:

- `label`: the real x-value of the glyph (sample)
- `image`: this is the binary data of the glyph itself
- (optional) `metadata`: this is an arbitrary `dict` containing some useful information the creator of the dataset wanted to embed into it

In [ ]:
loaded_split_0 = load_split_samples(
    dataset_path=c.dataset_path,
    split="0",
    desired_size=(64, 64),
    keep_alpha=True,
 )
loaded_split_1 = load_split_samples(
    dataset_path=c.dataset_path,
    split="1",
    desired_size=(64, 64),
    keep_alpha=True,
 )

samples_train = loaded_split_0
samples_test = loaded_split_1
images_train = [s.image for s in samples_train]
labels_train = [s.label for s in samples_train]
images_test = [s.image for s in samples_test]
labels_test = [s.label for s in samples_test]

# Augmentation

When training the neural network, we can _augment_ the samples, which is a fancy word for modifying them a bit to make the samples a bit harder for the neural network to learn from. In this simple case, we simply rotate them a bit and move them around a tiny bit. This way, the NN cannot rely on simple patterns like a single line through the image, but has to find other, more complex ways to predict the correct label.

# GlyphDataset

This class is a data structure that holds a series of samples and labels, and applies some kind of transformation to them. We need this class because PyTorch provides a `DataLoader` class whose constructor needs an instance of a `Dataset` class. By following PyTorch's recommended data loading procedures, we gain access to automatic optimizations which include caching, thus, making the training faster.

In [ ]:
BG_PATH = Path("data/img/pumpa.jpg")
CROP_SIZE = 64

background_image = cv2.imread(str(BG_PATH), cv2.IMREAD_COLOR)
if background_image is None:
    raise FileNotFoundError(f"Could not read background image from {BG_PATH}")


def put_image_on_random_background(image: np.ndarray) -> np.ndarray:
    h_bg, w_bg = background_image.shape[:2]
    if h_bg < CROP_SIZE or w_bg < CROP_SIZE:
        raise ValueError(
            f"Background image must be at least {CROP_SIZE}x{CROP_SIZE}, got {w_bg}x{h_bg}"
        )

    y = np.random.randint(0, h_bg - CROP_SIZE + 1)
    x = np.random.randint(0, w_bg - CROP_SIZE + 1)
    crop = background_image[y : y + CROP_SIZE, x : x + CROP_SIZE].astype(np.float32)

    glyph = image.astype(np.float32)

    if glyph.ndim == 2:
        glyph_rgb = np.repeat(glyph[:, :, None], 3, axis=2)
        alpha = np.ones((glyph.shape[0], glyph.shape[1], 1), dtype=np.float32)
        blended = glyph_rgb * alpha + crop * (1.0 - alpha)
    elif glyph.shape[2] == 4:
        glyph_rgb = glyph[:, :, :3]
        alpha = glyph[:, :, 3:4] / 255.0

        edge_mask = (alpha[:, :, 0] > 0.01) & (alpha[:, :, 0] < 0.99)
        if np.any(edge_mask):
            edge_rgb = glyph_rgb[edge_mask].mean(axis=1)
            edge_alpha = alpha[:, :, 0][edge_mask]
            ratio = edge_rgb / (edge_alpha * 255.0 + 1e-6)
            is_premultiplied = float(np.median(ratio)) < 0.75
        else:
            is_premultiplied = False

        if is_premultiplied:
            blended = glyph_rgb + crop * (1.0 - alpha)
        else:
            blended = glyph_rgb * alpha + crop * (1.0 - alpha)
    else:
        glyph_rgb = glyph[:, :, :3]
        alpha = np.ones((glyph.shape[0], glyph.shape[1], 1), dtype=np.float32)
        blended = glyph_rgb * alpha + crop * (1.0 - alpha)

    return np.clip(blended, 0, 255).astype(np.uint8)


affine = A.Affine(
    rotate=(-1, 1),
    translate_percent={"x": (-0.01, 0.01), "y": (-0.01, 0.01)},
    fit_output=False,
    keep_ratio=True,
    border_mode=cv2.BORDER_CONSTANT,
    fill=255,
    p=1.0,
)
hue_shift = A.HueSaturationValue(
    hue_shift_limit=(0, 0),
    p=1.0,
)
additive_noise = A.GaussNoise(
    std_range=(0.003, 0.01),
    mean_range=(0.0, 0.0),
    per_channel=False,
    p=1.0,
)

normalize = A.Normalize(mean=0.0, std=1.0, max_pixel_value=255.0)
to_tensor = ToTensorV2()
pipeline = A.Compose([affine, hue_shift, additive_noise, normalize, to_tensor], seed=c.seed)
normalize_pipeline = A.Compose([normalize, to_tensor])


def affine_and_normalize(image: np.ndarray) -> torch.Tensor:
    image_with_bg = put_image_on_random_background(image)
    return pipeline(image=image_with_bg)["image"]


def just_normalize(image: np.ndarray) -> torch.Tensor:
    rgb_image = image[:, :, :3] if image.ndim == 3 and image.shape[2] == 4 else image
    return normalize_pipeline(image=rgb_image)["image"]


dataset_train = GlyphDataset(
    name="Training",
    images=images_train,
    labels=labels_train,
    transform=affine_and_normalize if c.augment else just_normalize,
)
dataset_test = GlyphDataset(
    name="Testing",
    images=images_test,
    labels=labels_test,
    transform=just_normalize,
)

print(f"train dataset size: {len(dataset_train)}")
print(f"test dataset size: {len(dataset_test)}")

In [ ]:
show_datasets(dataset_train, dataset_test, n_samples=10)

# Model

Here, we initialize the model and other important objects required during training. The DataLoader is also created here. The reason why it is not created right after the creation of the `GlyphDataset` is because we want to re-create the DataLoader when we want to re-create the model from scratch. The `GlyphDataset` never changes, it holds no mutable state. However, the `DataLoader` needs a seed which feeds a RNG inside the `DataLoader`. If we want to deterministically re-create the training, we need to reset the `DataLoader`.

The `scheduler` is an object responsible for the changing of the learning rate during training.

In [ ]:
device = os.environ["MGML_DEVICE"]
print(f"Training device: {device}")

model = BinnedGlyphRegressor(num_divisions=c.num_divisions)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=c.learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=c.scheduler_steps, gamma=c.scheduler_gamma)
generator = torch.Generator().manual_seed(c.seed)

data_loader_train = DataLoader(
    dataset_train,
    batch_size=c.batch_size,
    num_workers=c.data_loader_num_workers,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True,
    generator=generator,
)
data_loader_test = DataLoader(
    dataset_test,
    batch_size=c.batch_size,
    num_workers=c.data_loader_num_workers,
    pin_memory=True,
    persistent_workers=True,
)

# Training

The training is done in _steps_. One step corresponds to a single batch processed by the NN (with the update of the NN's weights). In this example, we don't even have epochs. Everything is done in steps. At the beginning of the cell, we create a `train_iterator`, which is an iterator that can fetch training samples from the training `DataLoader`. Once this iterator is exhausted, it's reset to the beginning and looped through again. This way, we can train indefinitely and the learning curve is not dependent on the amount of samples we're training on. However, it's still dependent on the batch size. If we increase the batch side, the number of steps required to go through the entire training set gets lower, because we're crunching through the dataset at a quicker rate.

In [ ]:
from collections import deque

model.to(device)

step = 0
running_loss_train = 0.0
running_error_train = 0.0
num_batches_train = 0

recent_train_losses = deque(maxlen=20)

train_iterator = iter(data_loader_train)

while step < c.max_steps:
    model.train()

    try:
        inputs, labels = next(train_iterator)
    except StopIteration:
        train_iterator = iter(data_loader_train)
        inputs, labels = next(train_iterator)

    inputs = inputs.to(device, non_blocking=True).float()
    labels = labels.to(device, non_blocking=True).float().view(-1)

    optimizer.zero_grad()
    logits: torch.Tensor = model(inputs)
    preds = model.logits_to_labels(logits)

    loss = criterion(preds, labels)
    loss.backward()
    optimizer.step()
    scheduler.step()

    error = torch.mean(torch.abs(preds - labels)).item()

    running_loss_train += loss.item()
    running_error_train += error
    num_batches_train += 1
    recent_train_losses.append(loss.item())
    step += 1

    if step % 100 == 0 or step == c.max_steps:
        avg_recent_loss = sum(recent_train_losses) / len(recent_train_losses)
        print("=" * 80)
        print(
            f"Step {step}: loss(last {len(recent_train_losses)} avg)={avg_recent_loss:.6f}, "
            f"lr={scheduler.get_last_lr()[0]:.6f}"
        )

Here, we show a graph which shows the real value of x vs. the predicted one. All the points on this graph should fall as close to the $y = x$ line as possible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

show_truth_vs_pred_graph(
    title="Train: Ground Truth vs Predicted x (random 1000 samples)",
    n_samples=1000,
    model=model,
    dataset=dataset_train,
    device=device,
    ax=axes[0],
)

show_truth_vs_pred_graph(
    title="Test: Ground Truth vs Predicted x (random 1000 samples)",
    n_samples=1000,
    model=model,
    dataset=dataset_test,
    device=device,
    ax=axes[1],
)

fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

fig, _, train_worst_5_loss_avg, train_worst_5_losses = show_loss_vs_x_graph(
    title="Train: Loss vs. x (random 5000 samples)",
    n_samples=5000,
    model=model,
    dataset=dataset_train,
    device=device,
    loss_fn=criterion,
    ax=axes[0],
)

fig, _, test_worst_5_loss_avg, test_worst_5_losses = show_loss_vs_x_graph(
    title="Test: Loss vs. x (random 5000 samples)",
    n_samples=5000,
    model=model,
    dataset=dataset_test,
    device=device,
    loss_fn=criterion,
    ax=axes[1],
)

fig.tight_layout()
plt.show()

task.logger.report_scalar(
    title="Evaluation",
    series="Train Worst 5 Loss Avg",
    value=train_worst_5_loss_avg,
    iteration=c.max_steps,
)
task.logger.report_scalar(
    title="Evaluation",
    series="Test Worst 5 Loss Avg",
    value=test_worst_5_loss_avg,
    iteration=c.max_steps,
)

print(f"Train worst 5 losses: {train_worst_5_losses}")
print(f"Train worst 5 average loss: {train_worst_5_loss_avg:.6f}")
print(f"Test worst 5 losses: {test_worst_5_losses}")
print(f"Test worst 5 average loss: {test_worst_5_loss_avg:.6f}")

In [ ]:
task.flush(wait_for_uploads=True)
task.close()